# What cache budget would cover this trace?

Find the smallest page budget that reuses a chosen fraction of complete prefix blocks in 1,375 FreeInference requests. The default target is 80%. Change it and rerun the last two code cells to compare budgets.

Prefix Atlas computed the exact capacity curve from Harvard MaDSys’s May 17, 2026 root-session trace. This notebook reads the release’s curve and evaluates its integer breakpoints with Python’s standard library. The model runs requests serially in timestamp order against one fully associative LRU cache. It counts referenced full 16-token pages. Memory for partial pages, decoding, pins, and concurrent requests is outside this model.

[Trace and transformation](https://github.com/pgg898200-ship-it/cachepin/blob/dfb0d5f90dadae3a6ca3a910923b9178724b9505/tools/prefix-atlas/trace_sources.md) · [FreeInference dataset, Harvard MaDSys](https://huggingface.co/datasets/harvardMadsys/freeinference_agentic_trace) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

In [1]:
import hashlib
import sys

async def public_bytes(url):
    if sys.platform == "emscripten":
        from pyodide.http import pyfetch
        response = await pyfetch(url)
        if not response.ok:
            raise RuntimeError(f"HTTP {response.status}: {url}")
        return await response.bytes()
    from urllib.request import urlopen
    with urlopen(url, timeout=30) as response:
        return response.read()

from html.parser import HTMLParser
import csv
import io

class PublishedCurve(HTMLParser):
    def __init__(self):
        super().__init__()
        self.inside = False
        self.parts = []

    def handle_starttag(self, tag, attrs):
        if tag == "script":
            self.inside = dict(attrs).get("id") == "harvardData"

    def handle_data(self, data):
        if self.inside:
            self.parts.append(data)

    def handle_endtag(self, tag):
        if tag == "script":
            self.inside = False

url = "https://raw.githubusercontent.com/pgg898200-ship-it/cachepin/dfb0d5f90dadae3a6ca3a910923b9178724b9505/site/prefix-atlas/index.html"
parser = PublishedCurve()
parser.feed((await public_bytes(url)).decode("utf-8"))
curve_csv = "".join(parser.parts).strip()
assert hashlib.sha256(curve_csv.encode()).hexdigest() == "db9b50aa0194aea3987a3200aa586b9613c0766a5695f60a4f82b58d738bcd17"
curve = [{k: int(v) for k, v in row.items()} for row in csv.DictReader(io.StringIO(curve_csv))]
references = curve[0]["page_references"]
print(f"{len(curve):,} breakpoints · {references:,} page references")

1,563 breakpoints · 5,684,383 page references


## Reuse at a fixed budget

Between two breakpoints, each extra cache page adds the same number of hits. Locate the last breakpoint at or below the budget and apply its slope. Beyond the final breakpoint, the trace has exhausted its possible reuse.

In [2]:
from bisect import bisect_right

budgets = [row["capacity_pages"] for row in curve]

def hits_at(pages):
    if pages < 0:
        raise ValueError("The page budget must be nonnegative.")
    row = curve[bisect_right(budgets, pages) - 1]
    return row["hits"] + (pages - row["capacity_pages"]) * row["hits_per_additional_page"]

for pages in (4096, 8192, 16384):
    hits = hits_at(pages)
    print(f"{pages:>6,} pages   {hits:>9,} hits   {100 * hits / references:5.2f}%")

 4,096 pages   3,351,635 hits   58.96%
 8,192 pages   4,471,892 hits   78.67%
16,384 pages   4,919,957 hits   86.55%


## Find the smallest qualifying budget

Search for the first capacity that reaches the target and report the result one page below it. Exact integer comparisons matter when two neighboring budgets round to the same percentage. A cold page’s first reference always misses, so some targets are unreachable.

In [3]:
from fractions import Fraction

target_percent = "80"
target = Fraction(target_percent) / 100
if not 0 <= target <= 1:
    raise ValueError("Choose a target between 0 and 100 percent.")

def qualifies(pages):
    return hits_at(pages) * target.denominator >= references * target.numerator

if not qualifies(budgets[-1]):
    print(f"Unreachable. The trace tops out at {100 * hits_at(budgets[-1]) / references:.2f}% reuse.")
else:
    lo, hi = 0, budgets[-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if qualifies(mid):
            hi = mid
        else:
            lo = mid + 1
    print(f"First qualifying budget: {lo:,} pages.")
    for pages in range(max(0, lo - 1), lo + 1):
        print(f"{pages:,} pages   {hits_at(pages):,} hits   {100 * hits_at(pages) / references:.5f}%")

First qualifying budget: 8,822 pages.
8,821 pages   4,547,436 hits   79.99876%
8,822 pages   4,547,539 hits   80.00057%


A target here refers to the share of referenced prefix pages reused, weighted by page count. Request-level hit rates can differ. The 80% threshold belongs to this trace and cache policy. Run the [Prefix Atlas CLI](https://github.com/pgg898200-ship-it/cachepin/tree/main/tools/prefix-atlas#install-the-cli) on your own ordered request log before sizing a service. [Open the interactive calculator](https://pgg898200-ship-it.github.io/cachepin/prefix-atlas/)